# RZ42 rotary inverted pendulum (Furuta): equations of motion with Kane's method

This follows the same steps as `2dof_pandulum.ipynb`: frames → kinematic differential equations → partial velocities → $F_r$ and $F_r^*$ → $M_d\,\dot u + g_d = 0$ → state space → simulation → linearization.

**Bodies**
* **A** (arm): turns about the vertical motor axis `N.z` by $q_1$. It includes the arm print, the bearing housing, the 608ZZ outer ring, the AS5600 board, inserts, screws and the motor rotor.
* **B** (pendulum): turns about the arm axis `A.x` by $q_2$, on the 608ZZ bearing. $q_2 = 0$ hangs down and $q_2 = \pi$ is upright. It includes the print, the coin, the magnet and the 608 inner ring.

**Known from the CAD (numbers)**

| symbol | meaning | value |
|---|---|---|
| $r$ | motor axis → pendulum bearing centre, along the arm | 0.062 m |
| $h$ | arm rotation plane → pendulum axis height (not in the EOM, only used for drawing) | 0.014 m |
| $L_{arm}$ | arm length to the end face (drawing only) | 0.058 m |
| $L_{tip}$ | pivot → pendulum tip (drawing only) | 0.080 m |
| $g$ | gravity | 9.81 m/s² |

**Unknown (kept as symbols, to be estimated later)**

| symbol | meaning |
|---|---|
| $m_a,\ x_a,\ J_a$ | arm mass, horizontal COM distance from the motor axis, arm inertia about its COM around the vertical axis (rotor included) |
| $m_p,\ l_p,\ c_p$ | pendulum mass, COM distance below the pivot (along $-$`B.z`), COM offset along the hinge (`B.x`) |
| $I_{px}, I_{py}, I_{pz}, I_{pxz}$ | pendulum inertia about its COM in frame B (the coin makes $I_{pxz} \neq 0$) |
| $b_a,\ F_a$ | arm joint viscous damping and Coulomb friction |
| $b_p,\ F_p$ | pendulum (608ZZ) viscous damping and Coulomb friction |

Coulomb friction is smoothed as $F\tanh(u/v_s)$ so the ODE stays smooth; $v_s$ is a small chosen velocity, not a physical parameter.

In [ ]:
import sympy as sm
import sympy.physics.mechanics as me
import numpy as np
me.init_vprinting(use_latex='mathjax')

## 1. Symbols, frames, points

In [ ]:
# known
r, h, g = sm.symbols('r, h, g', positive=True)
v_s = sm.symbols('v_s', positive=True)            # tanh smoothing velocity for Coulomb friction
# unknown: arm
m_a, x_a, J_a = sm.symbols('m_a, x_a, J_a')
# unknown: pendulum
m_p, l_p, c_p = sm.symbols('m_p, l_p, c_p')
I_px, I_py, I_pz, I_pxz = sm.symbols('I_px, I_py, I_pz, I_pxz')
# unknown: joint friction / damping
b_a, F_a, b_p, F_p = sm.symbols('b_a, F_a, b_p, F_p')

N, A, B = sm.symbols('N, A, B', cls=me.ReferenceFrame)
O, Ao, P, Bo = sm.symbols('O, Ao, P, Bo', cls=me.Point)

q1, q2 = me.dynamicsymbols('q1, q2')
q = sm.Matrix([q1, q2])
A.orient_axis(N, q1, N.z)        # arm about the vertical motor axis
B.orient_axis(A, q2, A.x)        # pendulum about the arm axis (608 bearing)

O.set_vel(N, 0)                                   # motor axis on the arm plane
Ao.set_pos(O, x_a * A.x)                          # arm COM (horizontal distance only matters)
P.set_pos(O, r * A.x + h * A.z)                   # pendulum pivot = 608 centre
Bo.set_pos(P, c_p * B.x - l_p * B.z)              # pendulum COM, hanging below the pivot at q2 = 0

t = me.dynamicsymbols._t

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

# Visualization only: illustrative dimensions in meters
params = {
    r: 0.18,
    h: 0.04,
    x_a: 0.09,
    c_p: 0.0,
    l_p: 0.22,
}

def get_geometry(q1_deg, q2_deg):
    substitutions = {
        **params,
        q1: np.deg2rad(q1_deg),
        q2: np.deg2rad(q2_deg)
    }

    def pos(point):
        return np.array(
            point.pos_from(O).express(N).to_matrix(N)
            .subs(substitutions), dtype=float
        ).flatten()

    def axes(frame):
        return np.column_stack([
            np.array(v.express(N).to_matrix(N)
                     .subs(substitutions), dtype=float).flatten()
            for v in (frame.x, frame.y, frame.z)
        ])

    return {
        "O": np.zeros(3),
        "Ao": pos(Ao),
        "P": pos(P),
        "Bo": pos(Bo),
    }, {
        "N": np.eye(3),
        "A": axes(A),
        "B": axes(B),
    }

fig = plt.figure(figsize=(10, 9))
ax = fig.add_subplot(111, projection="3d")
plt.subplots_adjust(bottom=0.19)

def draw(q1_deg=35, q2_deg=30):
    ax.cla()
    points, frames = get_geometry(q1_deg, q2_deg)

    # Mechanical connections
    for start, end, color, name in [
        ("O", "P", "tab:blue", "Arm"),
        ("P", "Bo", "tab:orange", "Pendulum")
    ]:
        a, b = points[start], points[end]
        ax.plot(*zip(a, b), color=color, linewidth=4, label=name)

    # Points
    for name, p in points.items():
        ax.scatter(*p, color="black", s=35)
        ax.text(*(p + 0.012), name, fontsize=11)

    # Reference frames: RGB = XYZ
    for name, origin, scale in [
        ("N", points["O"], 0.11),
        ("A", points["O"], 0.095),
        ("B", points["P"], 0.085)
    ]:
        R = frames[name]
        for i, (color, axis_name) in enumerate(
            zip(["red", "green", "blue"], ["x", "y", "z"])
        ):
            direction = R[:, i]
            ax.quiver(
                *origin, *(scale * direction),
                color=color, arrow_length_ratio=0.15,
                linewidth=1.8
            )
            tip = origin + scale * direction
            ax.text(*tip, f"{name}.{axis_name}",
                    color=color, fontsize=9)

    ax.set(xlim=(-0.3, 0.3),
           ylim=(-0.3, 0.3),
           zlim=(-0.3, 0.3),
           xlabel="N.x (m)", ylabel="N.y (m)", zlabel="N.z (m)")
    ax.set_box_aspect([1, 1, 1])
    ax.view_init(elev=23, azim=-55)
    ax.set_title(f"Reference Frames | q1={q1_deg:.0f}°, q2={q2_deg:.0f}°")
    ax.legend(loc="upper left")
    fig.canvas.draw_idle()

draw()

ax_q1 = plt.axes([0.22, 0.09, 0.60, 0.025])
ax_q2 = plt.axes([0.22, 0.045, 0.60, 0.025])

s_q1 = Slider(ax_q1, "q1 (deg)", -180, 180, valinit=0)
s_q2 = Slider(ax_q2, "q2 (deg)", -180, 180, valinit=0)

def update(_):
    draw(s_q1.val, s_q2.val)

s_q1.on_changed(update)
s_q2.on_changed(update)

plt.show()


In [ ]:
T = me.dynamicsymbols('T')       # motor torque on the arm

## 2. Generalized speeds and kinematic differential equations

In [ ]:
u1, u2 = me.dynamicsymbols('u1, u2')
U = sm.Matrix([u1, u2])
U

In [ ]:
Ud = U.diff(t)
Ud

In [ ]:
qd = q.diff(t)
qd

In [ ]:
fk = sm.Matrix([
    q1.diff(t) - u1,
    q2.diff(t) - u2
])
Mk = fk.jacobian(qd)
qd_zero = [0 for qdi in qd]
qd0_repl = dict(zip(qd, qd_zero))
gk = fk.xreplace(qd0_repl)
qd_sol = -Mk.LUsolve(gk)
qd_repl = dict(zip(qd, qd_sol))
qd_repl

In [ ]:
qd_sol

## 3. Velocities and partial velocities

In [ ]:
N_V_Ao = Ao.vel(N).xreplace(qd_repl)
N_V_Bo = Bo.vel(N).xreplace(qd_repl)
N_W_A = A.ang_vel_in(N).xreplace(qd_repl)
N_W_B = B.ang_vel_in(N).xreplace(qd_repl)
vels = (N_V_Ao, N_V_Bo, N_W_A, N_W_B)
vels

In [ ]:
V_Ao, V_Bo, W_A, W_B = me.partial_velocity(vels, U, N)
V_Ao, V_Bo, W_A, W_B

## 4. Inertia dyadics
The arm only turns about `N.z = A.z`, so only its $zz$ inertia about its COM ($J_a$) appears. The other arm inertia terms only create bearing reaction loads.
The pendulum uses a full inertia about its COM in B, with the $xz$ product.

In [ ]:
I_A = J_a * A.z.outer(A.z)
I_B = (I_px * B.x.outer(B.x) + I_py * B.y.outer(B.y) + I_pz * B.z.outer(B.z)
       + I_pxz * (B.x.outer(B.z) + B.z.outer(B.x)))
I_B

## 5. Generalized active forces $F_r$
* Gravity acts along $-$`N.z`.
* Motor torque $T$ acts on the arm; it reacts on the fixed base, so it adds nothing more.
* Arm joint friction: $-(b_a u_1 + F_a\tanh(u_1/v_s))$ on A about `A.z`.
* Pendulum bearing friction: $-\tau_p$ on B about `A.x`, and $+\tau_p$ on A (action/reaction), with $\tau_p = b_p u_2 + F_p\tanh(u_2/v_s)$.

In [ ]:
R_Ao = -m_a * g * N.z
R_Bo = -m_p * g * N.z

tau_a = b_a * u1 + F_a * sm.tanh(u1 / v_s)
tau_p = b_p * u2 + F_p * sm.tanh(u2 / v_s)
T_A = (T - tau_a) * A.z + tau_p * A.x
T_B = -tau_p * A.x

Fr = []
for i in range(len(U)):
    Fr.append(V_Ao[i].dot(R_Ao) + V_Bo[i].dot(R_Bo) +
              W_A[i].dot(T_A) + W_B[i].dot(T_B))
Fr = sm.Matrix(Fr)
Fr

## 6. Generalized inertia forces $F_r^*$

In [ ]:
Rs_Ao = -m_a * Ao.acc(N).subs(qd_repl)
Rs_Bo = -m_p * Bo.acc(N).subs(qd_repl)
Ts_A = (-A.ang_acc_in(N).dot(I_A) - A.ang_vel_in(N).cross(I_A).dot(A.ang_vel_in(N))).subs(qd_repl)
Ts_B = (-B.ang_acc_in(N).dot(I_B) - B.ang_vel_in(N).cross(I_B).dot(B.ang_vel_in(N))).subs(qd_repl)

Frs = []
for i in range(len(U)):
    Frs.append(V_Ao[i].dot(Rs_Ao) + V_Bo[i].dot(Rs_Bo) +
               W_A[i].dot(Ts_A) + W_B[i].dot(Ts_B))
Frs = sm.Matrix(Frs)
Frs

## 7. Kane's equations $F_r + F_r^* = 0 \;\Rightarrow\; M_d\,\dot u + g_d = 0$

In [ ]:
fd = Fr + Frs
Md = fd.jacobian(Ud)
Md = sm.trigsimp(sm.expand(Md))
Md

In [ ]:
Ud0_repl = {udi: 0 for udi in Ud}
gd = fd.subs(Ud0_repl)
gd = sm.Matrix([sm.trigsimp(sm.expand(e)) for e in gd])
gd

In [ ]:
# sanity check: the hand-built equations equal sympy's own KanesMethod
kane = me.KanesMethod(N, q_ind=[q1, q2], u_ind=[u1, u2], kd_eqs=list(fk))
bodies = [me.RigidBody('arm', Ao, A, m_a, (I_A, Ao)),
          me.RigidBody('pendulum', Bo, B, m_p, (I_B, Bo))]
loads = [(Ao, R_Ao), (Bo, R_Bo), (A, T_A), (B, T_B)]
fr_k, frstar_k = kane.kanes_equations(bodies, loads)
sm.simplify(sm.expand(fr_k + frstar_k - fd))

## 8. Lumped (base) parameters
The 14 physical unknowns only enter the equations through **9 combinations**. These are what the experiments can identify:

$$J_0 = J_a + m_a x_a^2 + m_p (r+c_p)^2 + I_{pz},\quad J_1 = I_{py} - I_{pz} + m_p l_p^2,\quad J_2 = I_{px} + m_p l_p^2,$$
$$K = I_{pxz} + m_p l_p (r + c_p),\quad G = g\, m_p l_p,\quad b_a,\ F_a,\ b_p,\ F_p$$

With these the equations read

$$\begin{aligned}
(J_0 + J_1\sin^2 q_2)\,\dot u_1 + K\cos q_2\,\dot u_2 + 2J_1 \sin q_2\cos q_2\,u_1u_2 - K\sin q_2\,u_2^2 + b_a u_1 + F_a\tanh\tfrac{u_1}{v_s} &= T\\
K\cos q_2\,\dot u_1 + J_2\,\dot u_2 - J_1\sin q_2\cos q_2\,u_1^2 + G\sin q_2 + b_p u_2 + F_p\tanh\tfrac{u_2}{v_s} &= 0
\end{aligned}$$

The cell below checks this against the Kane result.

In [ ]:
J0, J1, J2, K, G = sm.symbols('J0, J1, J2, K, G')
lumped_defs = {
    J0: J_a + m_a*x_a**2 + m_p*(r + c_p)**2 + I_pz,
    J1: I_py - I_pz + m_p*l_p**2,
    J2: I_px + m_p*l_p**2,
    K:  I_pxz + m_p*l_p*(r + c_p),
    G:  g*m_p*l_p,
}
s2, c2 = sm.sin(q2), sm.cos(q2)
fd_lumped = -sm.Matrix([
    (J0 + J1*s2**2)*u1.diff(t) + K*c2*u2.diff(t) + 2*J1*s2*c2*u1*u2 - K*s2*u2**2
        + b_a*u1 + F_a*sm.tanh(u1/v_s) - T,
    K*c2*u1.diff(t) + J2*u2.diff(t) - J1*s2*c2*u1**2 + G*s2
        + b_p*u2 + F_p*sm.tanh(u2/v_s),
])
sm.simplify(sm.expand(fd_lumped.subs(lumped_defs) - fd))

## 9. State space $\;M_{ss}\,\dot x + g_{ss} = 0$, with $x = [q_1, q_2, u_1, u_2]$

In [ ]:
ss = sm.Matrix.vstack(q, U)
ssd = ss.diff(t)
g_ss = sm.Matrix.vstack(gk, gd)
M_ss = sm.diag(Mk, Md)
M_ss

In [ ]:
ssd_sol = -M_ss.LUsolve(g_ss)      # kept unsimplified: simplify() on this is slow and not needed
ssd_sol.shape

## 10. Numerical model
The **nominal values** come from the CAD mass model (PLA, 15 % infill, 2 perimeters, coin, magnet, bearing rings) and the MuJoCo file. They are only a **starting guess** for the parameter estimation.

In [ ]:
Q1, Q2, U1, U2, Tau = sm.symbols('Q1 Q2 U1 U2 Tau')
numeric_subs = {q1: Q1, q2: Q2, u1: U1, u2: U2, T: Tau}

params = [r, g, v_s, m_a, x_a, J_a, m_p, l_p, c_p, I_px, I_py, I_pz, I_pxz, b_a, F_a, b_p, F_p]

rhs_symbolic = ssd_sol.subs(numeric_subs)
rhs_numpy = sm.lambdify((Q1, Q2, U1, U2, Tau, *params), rhs_symbolic, 'numpy')

# known geometry
r_val, h_val, g_val = 0.062, 0.014, 9.81
L_arm, L_tip = 0.058, 0.080

# nominal guesses for the unknowns (CAD mass model)
p_nom = {
    r: r_val, g: g_val, v_s: 0.01,
    m_a: 0.03408, x_a: 0.03390, J_a: 1.9041e-5 + 6.8e-6,   # arm Izz about COM + motor rotor
    m_p: 0.01725, l_p: 0.04277, c_p: -0.00177,
    I_px: 1.5613e-5, I_py: 1.5204e-5, I_pz: 8.93e-7, I_pxz: 1.309e-6,
    b_a: 1.0e-4, F_a: 0.0,             # arm joint: depends on the driver; placeholder
    b_p: 1.744e-5, F_p: 3.457e-5,      # 608ZZ: from free-swing decay of the reference rig
}
p_vals = [p_nom[p] for p in params]

lumped_nom = {k: float(v.subs(p_nom)) for k, v in lumped_defs.items()}
lumped_nom

In [ ]:
qd1, qd2 = u1, u2
s2, c2 = sm.sin(q2), sm.cos(q2)
M_std = -Md
C_std = sm.Matrix([[ J1*s2*c2*qd2,  J1*s2*c2*qd1 - K*s2*qd2],
                   [-J1*s2*c2*qd1,  0]]).subs(lumped_defs)
G_std = sm.Matrix([0, G*s2]).subs(lumped_defs)
F_std = sm.Matrix([b_a*qd1 + F_a*sm.tanh(qd1/v_s), b_p*qd2 + F_p*sm.tanh(qd2/v_s)])
tau   = sm.Matrix([T, 0])

# should print Matrix([[0],[0]])
sm.simplify(sm.expand(M_std*Ud + C_std*U + G_std + F_std - tau + fd))

In [ ]:
# ---- natural frequencies from M(q) q'' + C q' + G(q) + F(q') = tau, linearized at an equilibrium ----
# tau = 0, C drops out (velocity^2 terms), G(q) ~ Ks*dq, F ~ D*dq (viscous part only)
for eq_name, q2_eq in [('hanging (q2 = 0)', 0), ('upright (q2 = pi)', sm.pi)]:
    eq = {q1: 0, q2: q2_eq}
    M0 = np.array(M_std.subs(eq).subs(p_nom), dtype=float)                 # mass matrix at equilibrium
    Ks = np.array(G_std.jacobian(q).subs(eq).subs(p_nom), dtype=float)     # gravity stiffness dG/dq
    D  = np.diag([float(b_a.subs(p_nom)), float(b_p.subs(p_nom))])         # viscous damping

    print(f'===== {eq_name} =====')
    # 1) arm free (motor off, T = 0): solve (Ks - w^2 M0) phi = 0  <=>  eig(M0^-1 Ks)
    w2, modes = np.linalg.eig(np.linalg.solve(M0, Ks))
    w2, modes = w2.real, modes.real                    # 2x2 real problem, drop the 0j parts
    for w2k, phi in zip(w2, modes.T):
        if w2k >= 0:
            wn = np.sqrt(w2k)
            print(f'arm free : wn = {wn:8.3f} rad/s = {wn/(2*np.pi):6.3f} Hz   mode [q1, q2] = {np.round(phi/np.abs(phi).max(), 3)}')
        else:
            print(f'arm free : unstable, grows like exp({np.sqrt(-w2k):.3f} t)   mode [q1, q2] = {np.round(phi/np.abs(phi).max(), 3)}')

    # 2) arm held by the stepper (q1 locked): only the pendulum row is left
    w2_lock = Ks[1, 1] / M0[1, 1]
    if w2_lock >= 0:
        print(f'arm held : wn = {np.sqrt(w2_lock):8.3f} rad/s = {np.sqrt(w2_lock)/(2*np.pi):6.3f} Hz')
    else:
        print(f'arm held : unstable, grows like exp({np.sqrt(-w2_lock):.3f} t)')

    # 3) with viscous damping (arm free): state space x = [dq, dq'], eigenvalues lambda
    A_lin = np.block([[np.zeros((2, 2)), np.eye(2)],
                      [-np.linalg.solve(M0, Ks), -np.linalg.solve(M0, D)]])
    lam = np.linalg.eigvals(A_lin)
    for l in lam:
        if l.imag > 0:                                # oscillating mode (print one of each pair)
            wn, zeta = abs(l), -l.real/abs(l)
            print(f'damped   : lambda = {l:.3f}  ->  wn = {wn:.3f} rad/s, wd = {l.imag/(2*np.pi):.3f} Hz, zeta = {zeta:.4f}')
        elif l.imag == 0:                             # real eigenvalue: decay (<0), rigid-body (0) or unstable (>0)
            print(f'damped   : lambda = {l.real:.3f}  (real)')
    print()

In [ ]:
def torque(t, state):
    """Motor torque on the arm. Zero = free motion (driver disabled)."""
    return 0.0

def dynamics(t, state):
    q1_val, q2_val, u1_val, u2_val = state
    result = rhs_numpy(q1_val, q2_val, u1_val, u2_val, torque(t, state), *p_vals)
    return np.asarray(result, dtype=float).reshape(-1)

def rk4_step(fun, t, state, dt):
    k1 = fun(t, state)
    k2 = fun(t + dt/2, state + dt*k1/2)
    k3 = fun(t + dt/2, state + dt*k2/2)
    k4 = fun(t + dt, state + dt*k3)
    return state + dt*(k1 + 2*k2 + 2*k3 + k4)/6

def simulate(fun, x0, t_end, dt):
    times = np.arange(0.0, t_end + dt, dt)
    states = np.empty((times.size, len(x0)))
    states[0] = x0
    for k in range(times.size - 1):
        states[k + 1] = rk4_step(fun, times[k], states[k], dt)
    return times, states

### 10a. Free motion: released 5° from upright, driver off ($T = 0$)

In [ ]:
initial_state = np.array([
    0.0,                         # q1, arm
    np.deg2rad(180.0 - 5.0),     # q2, pendulum (180 deg = upright)
    0.0,                         # u1
    0.0,                         # u2
])
dt = 0.001
times, states = simulate(dynamics, initial_state, 8.0, dt)
states.shape

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax[0].plot(times, np.rad2deg(states[:, 0]), color='#2a6fdb')
ax[0].set_ylabel('arm q1 [deg]')
ax[1].plot(times, np.rad2deg(states[:, 1]), color='#d1495b')
ax[1].set_ylabel('pendulum q2 [deg]')
ax[1].set_xlabel('time [s]')
for a in ax: a.grid(True, alpha=0.3)
ax[0].set_title('Free motion from 5° off upright, T = 0')
plt.tight_layout(); plt.show()

### 10b. Prescribed arm motion (how the closed-loop stepper is really driven)
The ZDT driver follows a position command, so the arm angle is effectively the **input**: $q_1(t),\ \dot q_1(t),\ \ddot q_1(t)$ are known.
Then only the second Kane equation is integrated:
$$\dot u_2 = -\frac{M_{d,21}\,\ddot q_1 + g_{d,2}}{M_{d,22}}$$
This also gives the torque the motor must deliver, from the first equation.
The command is the same one used in MuJoCo (`run_sine.py`): $q_1 = A\,e(t)\sin(2\pi f t)$ with a 1 s cosine fade-in $e(t)$.

In [ ]:
Q1dd = sm.symbols('Q1dd')
acc_subs = {u1.diff(t): Q1dd}
u2d_expr = (-(Md[1, 0]*u1.diff(t) + gd[1]) / Md[1, 1]).subs(acc_subs).subs(numeric_subs)
u2d_numpy = sm.lambdify((Q1, Q2, U1, U2, Q1dd, *params), u2d_expr, 'numpy')
# motor torque needed, from row 1 with the computed u2d
U2d = sm.symbols('U2d')
# fd[0] = T - (rest), so T = T - fd[0]  (no sm.solve: it is very slow on this expression)
T_expr = (T - fd[0]).subs({u1.diff(t): Q1dd, u2.diff(t): U2d}).subs(numeric_subs)
T_numpy = sm.lambdify((Q1, Q2, U1, U2, Q1dd, U2d, *params), T_expr, 'numpy')

def arm_command(t, amp, freq, ramp=1.0):
    """q1, q1dot, q1ddot of the MuJoCo sine command (cosine fade-in over `ramp` s)."""
    w = 2*np.pi*freq
    if t < ramp:
        e   = 0.5*(1 - np.cos(np.pi*t/ramp))
        ed  = 0.5*np.pi/ramp*np.sin(np.pi*t/ramp)
        edd = 0.5*(np.pi/ramp)**2*np.cos(np.pi*t/ramp)
    else:
        e, ed, edd = 1.0, 0.0, 0.0
    s, c = np.sin(w*t), np.cos(w*t)
    return (amp*e*s,
            amp*(ed*s + e*w*c),
            amp*(edd*s + 2*ed*w*c - e*w*w*s))

def make_prescribed(amp, freq):
    def fun(t, x):
        q2v, u2v = x
        a, ad, add = arm_command(t, amp, freq)
        return np.array([u2v, float(u2d_numpy(a, q2v, ad, u2v, add, *p_vals))])
    return fun

def run_prescribed(amp, freq, t_end, dt=0.001):
    times, x = simulate(make_prescribed(amp, freq), np.array([0.0, 0.0]), t_end, dt)
    cmd = np.array([arm_command(tk, amp, freq) for tk in times])
    u2d = np.array([u2d_numpy(cmd[k, 0], x[k, 0], cmd[k, 1], x[k, 1], cmd[k, 2], *p_vals) for k in range(times.size)])
    Tm = np.array([T_numpy(cmd[k, 0], x[k, 0], cmd[k, 1], x[k, 1], cmd[k, 2], u2d[k], *p_vals) for k in range(times.size)])
    return times, cmd, x, Tm

In [ ]:
# Compare with MuJoCo (CSV files written by run_sine.py, copied next to this notebook)
import os
cases = [(0.3, 1, 'demo_A0.3_f1.0.csv'), (0.3, 1.9, 'demo_A0.3_f1.9_resonance.csv')]

fig, axes = plt.subplots(len(cases), 1, figsize=(9, 6), sharex=True)
for ax, (amp, freq, fname) in zip(axes, cases):
    fname = os.path.join('..', 'dataset', 'nominal_model_sine', fname)
    tk, cmd, x, Tm = run_prescribed(amp, freq, 12.0)
    ax.plot(tk, np.rad2deg(x[:, 0]), color='#d1495b', label='Kane model (pendulum)')
    if os.path.exists(fname):
        mj = np.genfromtxt(fname, delimiter=',', names=True)
        ax.plot(mj['t'], np.rad2deg(mj['pendulum']), '--', color='#333333', lw=1, label='MuJoCo (pendulum)')
    ax.plot(tk, np.rad2deg(cmd[:, 0]), color='#2a6fdb', lw=1, alpha=0.6, label='arm command')
    ax.set_ylabel('angle [deg]')
    ax.set_title(f'arm sine A = {amp} rad, f = {freq} Hz;  peak motor torque {np.abs(Tm).max()*1e3:.1f} mN·m')
    ax.grid(True, alpha=0.3)
axes[0].legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('time [s]'); axes[-1].set_xlim(0, 12)
plt.tight_layout(); plt.show()

# keep the 1.9 Hz run for the animation
anim_t, anim_cmd, anim_x, _ = run_prescribed(0.3, 1.9, 8.0)

## 11. 3-D animation (prescribed 1.9 Hz arm sine)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 60  # MB

def geometry(q1v, q2v):
    """Points in N: motor axis top, arm end, pivot, pendulum tip."""
    ax_ = np.array([np.cos(q1v), np.sin(q1v), 0.0])          # A.x
    ay_ = np.array([-np.sin(q1v), np.cos(q1v), 0.0])         # A.y
    az_ = np.array([0.0, 0.0, 1.0])
    O_ = np.zeros(3)
    end_ = L_arm * ax_ + h_val * az_
    piv_ = r_val * ax_ + h_val * az_
    bz_ = -np.sin(q2v) * ay_ + np.cos(q2v) * az_              # B.z in N
    tip_ = piv_ - L_tip * bz_
    return O_, end_, piv_, tip_

frame_stride = max(1, int(round(0.04 / (anim_t[1] - anim_t[0]))))   # 25 fps
frames = np.arange(0, anim_t.size, frame_stride)

fig = plt.figure(figsize=(6, 6))
ax3 = fig.add_subplot(111, projection='3d')
lim = 0.1
ax3.set_xlim(-lim, lim); ax3.set_ylim(-lim, lim); ax3.set_zlim(-0.08, 0.12)
ax3.set_xlabel('x [m]'); ax3.set_ylabel('y [m]'); ax3.set_zlabel('z [m]')
ax3.plot([0, 0], [0, 0], [-0.08, 0], color='grey', lw=6, alpha=0.4)   # motor shaft
arm_line, = ax3.plot([], [], [], '-', color='#2a6fdb', lw=5)
pen_line, = ax3.plot([], [], [], 'o-', color='#d1495b', lw=3, markersize=4)
txt = ax3.text2D(0.03, 0.95, '', transform=ax3.transAxes)

def update(k):
    O_, end_, piv_, tip_ = geometry(anim_cmd[k, 0], anim_x[k, 0])
    arm_line.set_data([O_[0], end_[0]], [O_[1], end_[1]]); arm_line.set_3d_properties([h_val, end_[2]])
    pen_line.set_data([piv_[0], tip_[0]], [piv_[1], tip_[1]]); pen_line.set_3d_properties([piv_[2], tip_[2]])
    txt.set_text(f't = {anim_t[k]:.2f} s')
    return arm_line, pen_line, txt

anim = FuncAnimation(fig, update, frames=frames, interval=40, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

## 12. Linearization about upright ($q_2 = \pi$)
Coulomb friction is left out here ($F_a = F_p = 0$), because the slope of $\tanh$ at $u = 0$ ($F/v_s$) is an artefact of the smoothing, not real damping.

In [ ]:
x_sym = sm.Matrix([q1, q2, u1, u2])
A_sym = ssd_sol.jacobian(x_sym)
B_sym = ssd_sol.jacobian(sm.Matrix([T]))

parameter_values = dict(p_nom)
parameter_values.update({F_a: 0.0, F_p: 0.0, q1: 0, q2: sm.pi, u1: 0, u2: 0, T: 0, h: h_val})

A_num = np.array(A_sym.subs(parameter_values), dtype=float)
B_num = np.array(B_sym.subs(parameter_values), dtype=float)
print("A =\n", A_num)
print("B =\n", B_num)

f_eq = np.array(ssd_sol.subs(parameter_values), dtype=float).reshape(-1)
print("f(x_eq, u_eq) =", f_eq)

In [ ]:
def ctrb(A_, B_):
    n = A_.shape[0]
    return np.hstack([np.linalg.matrix_power(A_, k) @ B_ for k in range(n)])

Ctrb = ctrb(A_num, B_num)
print("Controllability rank (torque input):", np.linalg.matrix_rank(Ctrb))
print("Open-loop eigenvalues:", np.linalg.eigvals(A_num))

### Same linearization with the **arm acceleration** as the input
The stepper is commanded in position, speed or acceleration, not torque. Taking $a = \ddot q_1$ as the input, the pendulum row gives
$$\dot u_2 = -\frac{M_{d,21}\,a + g_{d,2}}{M_{d,22}}$$
This is usually the model you design the balancing controller with.

In [ ]:
a_in = sm.symbols('a')
f_acc = sm.Matrix([u1, u2, a_in, (-(Md[1, 0]*a_in + gd[1]) / Md[1, 1])])
A2_num = np.array(f_acc.jacobian(x_sym).subs(parameter_values), dtype=float)
B2_num = np.array(f_acc.jacobian(sm.Matrix([a_in])).subs(parameter_values).subs(a_in, 0), dtype=float)
print("A_acc =\n", A2_num)
print("B_acc =\n", B2_num)
print("Controllability rank (acceleration input):", np.linalg.matrix_rank(ctrb(A2_num, B2_num)))
print("Eigenvalues:", np.linalg.eigvals(A2_num))

In [ ]:
k1, k2, k3, k4 = sm.symbols('k1, k2, k3, k4')
K_gain = -1 * sm.Matrix([k1, k2, k3, k4]).transpose()
A_cl = sm.Matrix(A2_num) + sm.Matrix(B2_num) * K_gain
A_cl

## 13. Linear-in-parameters form (link to the identification)
Kane's equations are **linear in the 9 lumped parameters** $\theta = [J_0, J_1, J_2, K, G, b_a, F_a, b_p, F_p]^T$:
$$Y(q, u, \dot u)\,\theta = \begin{bmatrix} T \\ 0\end{bmatrix}$$
On the RZ42 rig the motor torque $T$ is **not measured**: a stepper driver cannot report it reliably. So only the **pendulum row** (row 2, right-hand side 0) is usable. It is homogeneous, which means it fixes the parameters only as ratios to $J_2$:
$$\kappa = K/J_2,\quad \rho = J_1/J_2,\quad \omega^2 = G/J_2,\quad \beta = b_p/J_2,\quad \phi = F_p/J_2$$
These are all that a stepper-based (arm-acceleration-input) controller needs.
* Free swings with the arm held identify $\omega^2, \beta, \phi$.
* Arm-driven runs (sines, chirp, swings) add $\kappa$ and $\rho$.
* The absolute scale $J_2$ comes from free decays with a known added mass.
* $J_0$, $b_a$, $F_a$ (row 1) are not identifiable without torque.

See `02_parameter_identification.ipynb`.

In [ ]:
theta = sm.Matrix([J0, J1, J2, K, G, b_a, F_a, b_p, F_p])
Y = (-fd_lumped).jacobian(theta)          # regressor; -fd_lumped = Y*theta - [T, 0]
check = sm.simplify(sm.expand(Y*theta - sm.Matrix([T, 0]) + fd_lumped))
Y, check